In [11]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [7]:
def generate_joke(state:JokeState):

    prompt=f"generate a joke on the topic {state['topic']}"
    response=llm.invoke(prompt).content

    return {'joke':response}

In [8]:
def explain_joke(state:JokeState):

    prompt=f"Write an explanation for the joke {state['joke']}"
    response=llm.invoke(prompt).content

    return {'explanation':response}

In [12]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('explain_joke',explain_joke)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','explain_joke')
graph.add_edge('explain_joke',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)


In [13]:
config1={'configurable' :{'thread_id':'1'}}
workflow.invoke({'topic':'Pizza'},config=config1)

{'topic': 'Pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or stressed.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase\'s usual meaning of being irritable. The joke relies on this wordplay to create humor, as it\'s unexpected and clever to use a phrase that\'s usually applied to people to describe a food item like pizza. The punchline "it was feeling a little crusty" is a clever and amusing way to explain why the pizza is in a bad mood, making it a lighthearted and amusing joke.'}

In [14]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or stressed.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase\'s usual meaning of being irritable. The joke relies on this wordplay to create humor, as it\'s unexpected and clever to use a phrase that\'s usually applied to people to describe a food item like pizza. The punchline "it was feeling a little crusty" is a clever and amusing way to explain why the pizza is in a bad mood, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'threa

In [15]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or stressed.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to both the pizza\'s crust and the phrase\'s usual meaning of being irritable. The joke relies on this wordplay to create humor, as it\'s unexpected and clever to use a phrase that\'s usually applied to people to describe a food item like pizza. The punchline "it was feeling a little crusty" is a clever and amusing way to explain why the pizza is in a bad mood, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thre

## Time Travel

In [16]:
workflow.get_state({'configurable':{'thread_id':'1','checkpoint_id': '1f1649e8-1b08-6b5c-8000-58167f26c2ff'}})

StateSnapshot(values={'topic': 'Pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1649e8-1b08-6b5c-8000-58167f26c2ff'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-10T07:32:16.863700+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1649e8-1b01-63c1-bfff-70bc7b8d9fb3'}}, tasks=(PregelTask(id='4131b3a5-db93-45bb-98ae-30138d648406', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.'}),), interrupts=())

In [18]:
workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id': '1f1649e8-1b08-6b5c-8000-58167f26c2ff'}})

{'topic': 'Pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or stressed.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to the pizza\'s crust, but also to the phrase "feeling a little crusty" which means being in a bad mood. The joke relies on this wordplay to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The result is a lighthearted and amusing joke that uses language in a creative way.'}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or grumpy, often due to being tired, hungry, or stressed.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to the pizza\'s crust, but also to the phrase "feeling a little crusty" which means being in a bad mood. The joke relies on this wordplay to create a humorous connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty"). The result is a lighthearted and amusing joke that uses language in a creative way.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'c